# Дообучение PERSON-распознавателя (ruBERT) для модуля анонимизации

Обучаем русский BERT как **token-classifier** распознавать ФИО (PERSON) на нашем размеченном датасете.
На выходе — папка с моделью, которую модуль подхватывает через `PERSON_MODEL_DIR`.

**Порядок:** `Среда выполнения → Сменить среду → GPU (T4)`, затем выполнять ячейки сверху вниз.

**Нужные файлы из репозитория** (`data/training/`):
- `train.conll`, `dev.conll`, `test.conll` — основной датасет;
- `train_negatives.conll`, `dev_negatives.conll` — негативы против ложных срабатываний
  (топонимы и частые слова размечены как `O`; генерируются `augment_negatives.py`).

## 0. Проверка GPU

In [ ]:
!nvidia-smi

## 1. Установка зависимостей

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.19" seqeval accelerate

## 2. Загрузка данных
Загрузите `train.conll`, `dev.conll`, `test.conll` и негативы
`train_negatives.conll`, `dev_negatives.conll` (кнопкой ниже).
Негативы опциональны: без них обучение пойдёт только на основном наборе, но
ложные срабатывания (город/частое слово как ФИО) не уменьшатся.

In [ ]:
from google.colab import files
print('Выберите train/dev/test.conll + (опц.) train_negatives.conll, dev_negatives.conll')
uploaded = files.upload()
print('Загружены:', list(uploaded.keys()))

## 3. Разбор CoNLL и подготовка меток
Оставляем только **PERSON**: все прочие типы (INN/PHONE/…) схлопываем в `O` — их в модуле ловит regex, а трансформер отвечает за ФИО. Метки: `O`, `B-PERSON`, `I-PERSON`.

In [ ]:
import os

def read_conll(path):
    sents, toks, tags = [], [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if toks:
                    sents.append((toks, tags)); toks, tags = [], []
                continue
            parts = line.split('\t')
            token, tag = parts[0], parts[-1]
            # оставляем только PERSON, остальное -> O
            tag = tag if tag.endswith('PERSON') else 'O'
            toks.append(token); tags.append(tag)
    if toks:
        sents.append((toks, tags))
    return sents

def read_with_negatives(base, neg):
    sents = read_conll(base)
    if os.path.exists(neg):
        extra = read_conll(neg)
        sents = sents + extra
        print(f'  + {len(extra)} негативов из {neg}')
    else:
        print(f'  (нет {neg} — обучение без негативов)')
    return sents

train = read_with_negatives('train.conll', 'train_negatives.conll')
dev   = read_with_negatives('dev.conll', 'dev_negatives.conll')
test  = read_conll('test.conll')
print(f'train={len(train)} dev={len(dev)} test={len(test)} предложений')

label_list = ['O', 'B-PERSON', 'I-PERSON']
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
n_person = sum(t.startswith('B-') for _, tg in train for t in tg)
print('PERSON-сущностей в train:', n_person)
print('пример:', train[1])

## 4. Датасеты и токенизация
Базовая модель — `cointegrated/rubert-tiny2` (быстрая, компактная). Для более высокого качества можно заменить на `ai-forever/ruBert-base` или `DeepPavlov/rubert-base-cased` (дольше учится, тяжелее модель).

In [ ]:
MODEL_NAME = 'cointegrated/rubert-tiny2'

from datasets import Dataset
from transformers import AutoTokenizer

def to_ds(sents):
    return Dataset.from_dict({
        'tokens': [t for t, _ in sents],
        'ner_tags': [[label2id[x] for x in tg] for _, tg in sents],
    })

ds_train, ds_dev, ds_test = to_ds(train), to_ds(dev), to_ds(test)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(batch):
    tok = tokenizer(batch['tokens'], truncation=True, is_split_into_words=True)
    labels = []
    for i, tags in enumerate(batch['ner_tags']):
        word_ids = tok.word_ids(batch_index=i)
        prev, lab = None, []
        for wid in word_ids:
            if wid is None:
                lab.append(-100)
            elif wid != prev:
                lab.append(tags[wid])
            else:
                # продолжение слова: B-PERSON -> I-PERSON, чтобы не плодить B
                t = tags[wid]
                lab.append(label2id['I-PERSON'] if id2label[t].endswith('PERSON') else t)
            prev = wid
        labels.append(lab)
    tok['labels'] = labels
    return tok

ds_train = ds_train.map(tokenize_and_align, batched=True, remove_columns=ds_train.column_names)
ds_dev   = ds_dev.map(tokenize_and_align,   batched=True, remove_columns=ds_dev.column_names)
ds_test  = ds_test.map(tokenize_and_align,  batched=True, remove_columns=ds_test.column_names)
print('OK, пример признаков:', {k: ds_train[0][k] for k in ('input_ids','labels')})

## 5. Модель, метрики и обучение

In [ ]:
import numpy as np
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)
import evaluate

seqeval = evaluate.load('seqeval')

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_pred, true_lab = [], []
    for pr, la in zip(preds, labels):
        tp, tl = [], []
        for pi, li in zip(pr, la):
            if li != -100:
                tp.append(id2label[pi]); tl.append(id2label[li])
        true_pred.append(tp); true_lab.append(tl)
    r = seqeval.compute(predictions=true_pred, references=true_lab)
    return {'precision': r['overall_precision'], 'recall': r['overall_recall'],
            'f1': r['overall_f1']}

args = TrainingArguments(
    output_dir='out', learning_rate=3e-5, num_train_epochs=8,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1',
    logging_steps=20, report_to='none')

trainer = Trainer(model=model, args=args, train_dataset=ds_train,
                  eval_dataset=ds_dev, processing_class=tokenizer,
                  data_collator=collator, compute_metrics=compute_metrics)
trainer.train()

## 6. Оценка на test (held-out)

In [ ]:
metrics = trainer.evaluate(ds_test)
print('TEST:', {k: round(v, 4) for k, v in metrics.items() if k.startswith('eval_')})

## 7. Быстрая проверка на живом примере

In [ ]:
from transformers import pipeline
nlp = pipeline('token-classification', model=model, tokenizer=tokenizer,
               aggregation_strategy='simple')
for s in ['прошу проверить инн клиента сергее павлове',
          'Я А.В. Морозова, мой снилс 093-248-086 78',
          'Договор подписал Пётр Иванович Смирнов']:
    ents = [(e['word'], round(e['score'],2)) for e in nlp(s) if e['entity_group'].endswith('PERSON')]
    print(s, '->', ents)

## 8. Сохранение и скачивание модели

In [ ]:
OUT_DIR = 'person_ruBERT'
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

import shutil
shutil.make_archive('person_ruBERT', 'zip', OUT_DIR)
from google.colab import files
files.download('person_ruBERT.zip')
print('Готово: скачан person_ruBERT.zip')

## 9. Как встроить в модуль (после скачивания)

1. Распаковать `person_ruBERT.zip` в проект, например в `models/person_ruBERT/` (внутри должен лежать `config.json`).
2. Доустановить в venv модуля: `pip install "transformers>=4.40" torch`.
3. Указать путь к модели и (пере)запустить сервис:
   ```bash
   export PERSON_MODEL_DIR=/abs/path/to/models/person_ruBERT
   ```
   Модуль сам подхватит модель вместо стоковой Natasha (см. `app/custom_recognizers.py`).
4. Прогнать **реальные** Тесты 3 и 4:
   ```bash
   PERSON_MODEL_DIR=/abs/.../person_ruBERT \
   GRAMLYNX_URL=http://127.0.0.1:8010 \
   python data/natasha_training/eval_2x2.py --cells 3,4 -K 5
   ```
   Harness проверяет наличие модели, поэтому цифры будут настоящими.